# Ticketmaster API Explorer
This notebook interacts with the Ticketmaster Discovery API to query events and venues in the UK.

## 1. Imports & Setup

In [ ]:
import requests
import os
from dotenv import load_dotenv
from pprint import pprint
import pandas as pd
# hi 

## 2. Configuration
Load the API key from the `.env` file and define base URL constants.

In [2]:
load_dotenv()

API_KEY = os.getenv("TICKET_API")

if not API_KEY:
    raise ValueError("That API key is not in your env, check your env for the correct variable name.")

BASE_URL = "https://app.ticketmaster.com"

EVENTS_ENDPOINT = "/discovery/v2/events"
VENUES_ENDPOINT = "/discovery/v2/venues"

print("Config loaded successfully.")

Config loaded successfully.


## 3. Events
Query the Ticketmaster Discovery API for events in the UK.

In [ ]:
api_key = API_KEY
all_events = []
current_page = 0
total_pages = 1 

while current_page < total_pages:
    events_url = f"{BASE_URL}{EVENTS_ENDPOINT}.json?apikey={api_key}&countryCode=GB&size=200&page={current_page}"
    events_response = requests.get(events_url).json()
    
    page_events = events_response.get('_embedded', {}).get('events', [])
    all_events.extend(page_events)
    
    total_pages = events_response.get('page', {}).get('totalPages', 1)
    current_page += 1

print(f"Successfully collected {len(all_events)} events total!")





# events_url = f"{BASE_URL}{EVENTS_ENDPOINT}"

# payload_events = {
#     "apikey": API_KEY,
#     "size": "200",
#     "countryCode": "GB"
# }

# response_event = requests.get(events_url, params=payload_events)
# response_event.raise_for_status()

# data_event = response_event.json()

# print(f"Status: {response_event.status_code}")

Status: 200


In [ ]:
# original

# uk_events = []

# event_list = data_event['_embedded']['events']

# for event in event_list:
#     event_data = {
#         "event_id":   event.get("id"),
#         "name": event.get("name"),
#         "date": event.get("dates", {}).get("start", {}).get("localDate"),
#         "time": event.get("dates", {}).get("start", {}).get("localTime"),
#         "multi_day_event": event.get("dates", {}).get("spanMultipleDays"),
#         "legal_age_enforced": event.get("ageRestrictions", {}).get("legalAgeEnforced"),
#         "category_segment": event.get("classifications", {})[0].get("segment", {}).get("name"),
#         "category_genre": event.get("classifications", {})[0].get("genre", {}).get("name"),
#         "category_sub_genre": event.get("classifications", {})[0].get("subGenre", {}).get("name"),
#         "all_inclusive_pricing": event.get("ticketing", {}).get("allInclusivePricing", {}).get("enabled"),
#         "venue_id": event.get("_embedded", {}).get("venues", {})[0].get("id"),
#         "venue_name": event.get("_embedded", {}).get("venues", {})[0].get("name"),
#         "address": event.get("_embedded", {}).get("venues", {})[0].get("address", {}).get("line1"),
#         "postcode": event.get("_embedded", {}).get("venues", {})[0].get("postalCode"),
#         "city": event.get("_embedded", {}).get("venues", {})[0].get("city", {}).get("name"),
#         "longitude": event.get("_embedded", {}).get("venues", {})[0].get("location", {}).get("longitude"),
#         "latitude": event.get("_embedded", {}).get("venues", {})[0].get("location", {}).get("latitude"),
#         "markets": event.get("_embedded", {}).get("venues", {})[0].get("markets")[0].get("name"),
#         "markets_id": event.get("_embedded", {}).get("venues", {})[0].get("markets")[0].get("id"),
#         "url": event.get("url"),

#     }

#     uk_events.append(event_data)

In [ ]:
# kiro 

uk_events = []

for event in all_events:
    class_list = event.get("classifications") or []
    cls = class_list[0] if class_list else {}
    
    venue_list = event.get("_embedded", {}).get("venues") or []
    vn = venue_list[0] if venue_list else {}
    
    market_list = vn.get("markets") or []
    mkt = market_list[0] if market_list else {}
    event_data = {
        "event_id": event.get("id"),
        "name": event.get("name"),
        "date": event.get("dates", {}).get("start", {}).get("localDate"),
        "time": event.get("dates", {}).get("start", {}).get("localTime"),
        "multi_day_event": event.get("dates", {}).get("spanMultipleDays"),
        "legal_age_enforced": event.get("ageRestrictions", {}).get("legalAgeEnforced"),
        
        "category_segment": cls.get("segment", {}).get("name"),
        "category_genre": cls.get("genre", {}).get("name"),
        "category_sub_genre": cls.get("subGenre", {}).get("name"),
        
        "all_inclusive_pricing": event.get("ticketing", {}).get("allInclusivePricing", {}).get("enabled"),
        
        "venue_id": vn.get("id"),
        "venue_name": vn.get("name"),
        "address": vn.get("address", {}).get("line1"),
        "postcode": vn.get("postalCode"),
        "city": vn.get("city", {}).get("name"),
        "longitude": vn.get("location", {}).get("longitude"),
        "latitude": vn.get("location", {}).get("latitude"),
        
        "markets": mkt.get("name"),
        "markets_id": mkt.get("id"),
        "url": event.get("url")
    }

    uk_events.append(event_data)

In [13]:
uk_events[0]

{'event_id': '17u8v0G6Cr7i1p0',
 'name': 'Philadelphia Eagles v Jacksonville Jaguars',
 'date': '2026-10-11',
 'time': '14:30:00',
 'multi_day_event': False,
 'legal_age_enforced': False,
 'category_segment': 'Sports',
 'category_genre': 'Football',
 'category_sub_genre': 'NFL',
 'all_inclusive_pricing': False,
 'venue_id': 'KovZ9177OxV',
 'venue_name': 'Tottenham Hotspur Stadium',
 'address': '782 High Rd',
 'postcode': 'N17 0BX',
 'city': 'London',
 'longitude': '-0.06787000',
 'latitude': '51.60081900',
 'markets': 'All of United Kingdom',
 'markets_id': '201',
 'url': 'https://www.eticketing.co.uk/nfl-tottenham?utm_source=Ticketmaster&utm_medium=Shell-Event&utm_campaign=cl:NFL-chl:Shell-Event-v.eagles-jaguars'}

In [6]:
print(len(uk_events))

200


In [ ]:
# implement pydantic validation

In [7]:
df_events = pd.DataFrame(uk_events)
df_events.head()

,event_id,name,date,time,multi_day_event,legal_age_enforced,category_segment,category_genre,category_sub_genre,all_inclusive_pricing,venue_id,venue_name,address,postcode,city,longitude,latitude,markets,markets_id,url
0,17u8v0G6Cr7i1p0,Philadelphia Eagles v Jacksonville Jaguars,2026-10-11,14:30:00,False,False,Sports,Football,NFL,False,KovZ9177OxV,Tottenham Hotspur Stadium,782 High Rd,N17 0BX,London,-0.06787000,51.60081900,All of United Kingdom,201,https://www.eticketing.co.uk/nfl-tottenham?utm...
1,17FYv0G65mKlspy,NFL London 2026: Houston Texans v Jacksonville...,2026-10-18,14:30:00,False,False,Sports,Football,NFL,False,KovZ9177ML0,Wembley Stadium,Wembley,HA9 0WS,London,-0.27958100,51.55780700,All of United Kingdom,201,https://www.eticketing.co.uk/jaguars/EDP/Event...
2,17u8v0G6CksBAD4,JAY-Z - 30,2026-09-04,17:00:00,False,False,Music,Hip-Hop/Rap,Trap,False,KovZ9177OxV,Tottenham Hotspur Stadium,782 High Rd,N17 0BX,London,-0.06787000,51.60081900,All of United Kingdom,201,https://www.ticketmaster.co.uk/jayz-30-london-...
3,17u8v0G6Cr7i1px,Indianapolis Colts v Washington Commanders,2026-10-04,14:30:00,False,False,Sports,Football,NFL,False,KovZ9177OxV,Tottenham Hotspur Stadium,782 High Rd,N17 0BX,London,-0.06787000,51.60081900,All of United Kingdom,201,https://www.eticketing.co.uk/nfl-tottenham?utm...
4,1AdjZbsGkiuENAI,Harry Potter and the Cursed Child - Parts 1 & ...,2026-07-10,14:00:00,False,False,Arts & Theatre,Theatre,Drama,False,KovZ9177gU0,Palace Theatre,"Cambridge Circus, Shaftesbury Avenue",W1D 8AY,London,-0.12970730,51.51316680,All of United Kingdom,201,https://theatre.ticketmaster.co.uk/book/17YYA-...


In [8]:
df_events.shape

(200, 20)

## 4. Venues
Query the Ticketmaster Discovery API for venues in the UK and extract key fields.

In [ ]:
# venues_url = f"{BASE_URL}{VENUES_ENDPOINT}"

# payload_venues = {
#     "apikey": API_KEY,
#     "countryCode": "GB",
#     "size": "200"
# }

# response_venue = requests.get(venues_url, params=payload_venues)
# response_venue.raise_for_status()

# data_venues = response_venue.json()

# print(f"Status: {response_venue.status_code}")


all_venues = []
current_page = 0
total_pages = 1 

while current_page < total_pages:
    venues_url = f"{BASE_URL}{VENUES_ENDPOINT}.json?apikey={api_key}&countryCode=GB&size=200&page={current_page}"
    venues_response = requests.get(venues_url).json()
    
    page_venues = venues_response.get('_embedded', {}).get('venues', [])
    all_venues.extend(page_venues)
    
    total_pages = venues_response.get('page', {}).get('totalPages', 1)
    current_page += 1

print(f"Successfully collected {len(all_venues)} venues total!")


Status: 200


In [ ]:
# uk_venues = []

# for venue in data_venues["_embedded"]["venues"]:
#     venue_data = {
#         "id":         venue.get("id"),
#         "name":       venue.get("name"),
#         "city":       venue.get("city", {}).get("name"),
#         "country":    venue.get("country", {}).get("name"),
#         "longitude":  venue.get("location", {}).get("longitude"),
#         "latitude":   venue.get("location", {}).get("latitude"),
#         "postcode": venue.get("postalCode")
#     }
#     uk_venues.append(venue_data)


uk_venues = []

for venue in all_venues:
    venue_data = {
        "id":         venue.get("id"),
        "name":       venue.get("name"),
        "city":       venue.get("city", {}).get("name"),
        "country":    venue.get("country", {}).get("name"),
        "longitude":  venue.get("location", {}).get("longitude"),
        "latitude":   venue.get("location", {}).get("latitude"),
        "postcode": venue.get("postalCode")
    }
    uk_venues.append(venue_data)

In [12]:
# Preview the first few venues
pprint(uk_venues[0])

{'city': 'Dorset',
 'country': 'Great Britain',
 'id': 'KovZ9177-2f',
 'latitude': '50.73191800',
 'longitude': '-2.75789500',
 'name': 'Bridport Electric Palace',
 'postcode': 'DT63NY'}


In [15]:
df_venues = pd.DataFrame(uk_venues)
df_venues.head()

,id,name,city,country,longitude,latitude,postcode
0,KovZ9177-2f,Bridport Electric Palace,Dorset,Great Britain,-2.75789500,50.73191800,DT63NY
1,KovZ9177-37,St Martin's Theatre,London,Great Britain,-0.12767200,51.51284800,WC2H 9NZ
2,KovZ9177-A0,The Spotlight,Hoddesdon,Great Britain,-0.01507500,51.75569100,EN11 8BE
3,KovZ9177-AV,sohoplace,London,Great Britain,-0.13043600,51.51560700,W1D 3BG
4,KovZ9177-E0,Accu Stadium,Huddersfield,Great Britain,-1.76892400,53.65464900,HD16PG
